### Ingestion pipeline / RAG preparation 

### Imports

In [1]:
import os
import sys
from pathlib import Path

import chromadb
import numpy as np
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))
from services import (
    html,
    text_patterns,
    text_processing,
)
from services.embedder_model_manager import EmbedderModelManager

/home/nikita/proga/TTRPG-Authentic-Personal-Assistant-TAPA/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Processing and chunking books 

In [ ]:
# Loading books data from HTML files.

data_path = Path("../data/books_sources/")

books_data = []

for book_dir in data_path.iterdir():
    book_name = book_dir.name
    print(f"Book: {book_dir.name}")

    files_data = html.load_html_files(book_dir)

    for file_data in files_data:
        books_data.append(
            Document(
                page_content=file_data[1],
                metadata={"title": file_data[0], "book_name": book_name},
            )
        )

Book: pf2e


load_html_files: 59548it [00:20, 2915.46it/s]


Book: starfinder1e


load_html_files: 0it [00:00, ?it/s]


In [ ]:
# Pasring, cleaning from redundant patterns and chunking text for every book.
# Straightforward approach until more complex logic is needed

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 300

text_splitter = RecursiveCharacterTextSplitter(
    separators=[" ", "\n", ""],
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
books_chunks = []

# Parsing and cleaning
parsed_texts = html.parse_html_data([x.page_content for x in books_data])
parsed_cleaned_texts = text_processing.clean_texts(
    parsed_texts,
    [x.metadata["book_name"] for x in books_data],
    text_patterns.books_patterns,
)

# Chunking with addiding to every chunk the title of original text
for elem, text in tqdm(
    zip(books_data, parsed_cleaned_texts),
    desc="chunk_text",
    total=len(parsed_cleaned_texts),
):
    books_chunks.extend(
        [
            Document(
                page_content=f"title: {elem.metadata['title']}\n[sep]\ncontent: {chunk}",
                metadata={"book_name": elem.metadata["book_name"]},
            )
            for chunk in text_splitter.split_text(text)
        ]
    )

chunk_text: 100%|██████████| 59548/59548 [00:12<00:00, 4765.46it/s]


In [ ]:
# Removing duplicates

_, ids = np.unique([x.page_content for x in books_chunks], return_index=True)
books_chunks = np.array(books_chunks)[ids]

print(len(books_chunks))
print(books_chunks[1])

82878
page_content='title: Actions
[sep]
content: (concentrate, manipulate) - Actions - Archives of Nethys: Pathfinder 2nd Edition Database

Actions
|
Activities
Activate
—Set Free
[two-actions]
(
concentrate
,
manipulate
);
Effect
You Release the weapon and it flutters through the air, fighting on its own against the last enemy you attacked, or the nearest enemy to it if your target has been defeated. At the end of your turn each round, the weapon can
Fly
up to its fly Speed of 40 feet, and then can either Fly again or Strike one creature within its reach.
The weapon has a space of 5 feet, but it doesn't block or impede enemies attempting to move though that space, nor does it benefit from or provide flanking. The weapon can't move through an enemy's space. The weapon can't use reactions, and its Fly actions don't trigger reactions.
While it's activated, an
animated
weapon makes Strikes with an attack modifier of +24 plus its item bonus to attack rolls. It uses the weapon's normal dam

In [ ]:
# import pickle

# with open("../data/books_chunks.pkl", "rb") as f:
#     books_chunks = pickle.load(f)

# with open("../data/embeddings.pkl", "rb") as f:
#     embeddings = pickle.load(f)

### Embedding chunks

In [ ]:
# Defining embedder

embedder = EmbedderModelManager()
embedder.load_model()

Loading model: intfloat/e5-large-v2


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 283.93it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-large-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Getting embeddings

BATCH_SIZE = 8

embeddings = []
for i in tqdm(
    range(0, len(books_chunks), BATCH_SIZE), desc="embedding chunks"
):
    batch_texts = [
        elem.page_content for elem in books_chunks[i : i + BATCH_SIZE]
    ]
    bacth_embeddings = embedder.embed_documents(batch_texts)
    embeddings.extend(bacth_embeddings)

### Creating VectorDB (ChromaDB)

In [ ]:
# Initializing Chromadb client, creating a new collection and saving vector_db

BATCH_SIZE = 2048

client = chromadb.PersistentClient(path="../data/vector_db")

collection = client.get_or_create_collection(name="books_collection")

start = collection.count()
ids = [str(start + i) for i in range(len(books_chunks))]

for i in tqdm(
    range(0, len(books_chunks), BATCH_SIZE), desc="filling ChromaDB"
):
    batch_chunks = books_chunks[i : i + BATCH_SIZE]
    batch_embs = embeddings[i : i + BATCH_SIZE]
    batch_ids = ids[i : i + BATCH_SIZE]
    collection.add(
        ids=batch_ids,
        documents=[d.page_content for d in batch_chunks],
        metadatas=[d.metadata for d in batch_chunks],
        embeddings=batch_embs,
    )


filling ChromaDB: 100%|██████████| 41/41 [01:49<00:00,  2.68s/it]


### Testing search

In [5]:
vector_db = Chroma(
    collection_name="books_collection",
    embedding_function=embedder,
    persist_directory="../data/vector_db",
)

/tmp/ipykernel_58030/2049171066.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


In [6]:
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 3,
        "filter": {"book_name": "pf2e"}
    },
)

In [8]:
docs = retriever.invoke("Invisibility cloak")
print(
    *[x.page_content[:1000] for x in docs],
    sep="\n--------------------------\n",
)

title: Spells
[sep]
content: Invisibility Cloak - Spells - Archives of Nethys: Pathfinder 2nd Edition Database

All Spells
Arcane
|
Divine
|
Elemental
|
Occult
|
Primal
Focus Spells
|
Rituals
There is a Legacy version
here
.
Invisibility Cloak
[two-actions]
Focus 4
Uncommon
Focus
Illusion
Manipulate
Wizard
Source
Player Core pg. 388
2.0
Duration
1 minute
You become invisible, with the same restrictions as the 2nd-rank
invisibility
spell.
Heightened (6th)
The duration increases to 10 minutes.
Heightened (8th)
The duration increases to 1 hour.
--------------------------
title: Spells
[sep]
content: Invisibility Cloak - Spells - Archives of Nethys: Pathfinder 2nd Edition Database

All Spells
Arcane
|
Divine
|
Elemental
|
Occult
|
Primal
Focus Spells
|
Rituals
There is a Remastered version
here
.
Invisibility Cloak
[two-actions]
Focus 4
Legacy Content
Uncommon
Illusion
Wizard
Source
Core Rulebook pg. 407
4.0
Cast
[two-actions]
somatic
Duration
1 minute
You become invisible, with the same r